# Online Retail — 5a, Câu 10: Luật kết hợp Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Tạo các luật từ tập phổ biến Apriori với `min_sup = 5%`, `min_conf = 50%`, kèm support, confidence và lift.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('../6451071041_LAB1_Bai2_Trang37') / 'Online Retail.xlsx', Path('../Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
germany = df.loc[df['Country'].eq('Germany')]
grouped = germany.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(germany['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in counts.items() if count >= minimum_count}
    all_frequent = dict(previous)
    frequent_items = set(counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda s: tuple(sorted(s)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            for parts in combinations(sorted(transaction & frequent_items), size):
                itemset = frozenset(parts)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        all_frequent.update(previous)
        size += 1
    return all_frequent
def association_rules(itemset_counts, transaction_count, min_confidence):
    rules = []
    for itemset, count in itemset_counts.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for parts in combinations(sorted(itemset), size):
                antecedent = frozenset(parts)
                consequent = itemset - antecedent
                confidence = count / itemset_counts[antecedent]
                if confidence >= min_confidence:
                    support = count / transaction_count
                    lift = confidence / (itemset_counts[consequent] / transaction_count)
                    rules.append({'antecedent': antecedent, 'consequent': consequent, 'support': support, 'confidence': confidence, 'lift': lift})
    return rules
min_support = 0.05
min_confidence = 0.50
frequent_counts = apriori_counts(transactions, min_support)
rules = association_rules(frequent_counts, len(transactions), min_confidence)
rules_df = pd.DataFrame([{'antecedent': ', '.join(sorted(r['antecedent'])), 'consequent': ', '.join(sorted(r['consequent'])), 'support': r['support'], 'confidence': r['confidence'], 'lift': r['lift']} for r in rules])
if not rules_df.empty:
    rules_df = rules_df.sort_values(['confidence', 'support'], ascending=False).reset_index(drop=True)
print(f'Minimum support: {min_support:.0%}; minimum confidence: {min_confidence:.0%}')
print(f'Số tập phổ biến: {len(frequent_counts):,}; số luật đạt ngưỡng: {len(rules_df):,}')
display(rules_df)